# Modeling

**Goal:** train baseline classifiers that predict employee attrition, and pick a leader to carry into tuning. The business cost of missing a leaver outweighs a false alarm, so **recall on the "Yes" (left) class is the priority metric**, with F1, ROC-AUC and PR-AUC as supporting evidence. Accuracy is deliberately ignored — a constant "No" classifier already scores ~84%.

Four families are compared, spanning the interpretable-to-powerful range: Logistic Regression, Random Forest, XGBoost, and LightGBM.

This stage stays scoped to model **selection**. Hyperparameter search and threshold tuning are Phase 6, the full evaluation (confusion matrix, ROC/PR curves) is Phase 7, and SHAP interpretation is Phase 8.

## 1. Setup

Standard imports plus the four estimators and the imbalanced-learn pipeline. `RANDOM_STATE = 42` matches every earlier stage, and `PROC` points at the `processed/` directory that the preprocessing and imbalance stages wrote to.

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
PROC = Path('../processed')

## 2. Load the modeling matrices

All inputs come from `processed/` — there is no in-memory hand-off between stage notebooks.

- `X_train` / `X_test` (1176 × 56 / 294 × 56) and their targets carry the real ~16% attrition rate.
- `X_train_smote` / `y_train_smote` (1972 × 56, balanced 50/50) are loaded **only for the final full-train fit of Logistic Regression**. They are never passed to cross-validation: doing so would let synthetic rows leak across folds and inflate the scores.

In [3]:
X_train = pd.read_parquet(PROC / 'X_train.parquet')
X_test = pd.read_parquet(PROC / 'X_test.parquet')
y_train = pd.read_parquet(PROC / 'y_train.parquet')['Attrition']
y_test = pd.read_parquet(PROC / 'y_test.parquet')['Attrition']

# SMOTE matrices are used only for the FINAL full-train fit of Logistic Regression.
# They are never fed to cross-validation (that would leak synthetic rows across folds).
X_train_smote = pd.read_parquet(PROC / 'X_train_smote.parquet')
y_train_smote = pd.read_parquet(PROC / 'y_train_smote.parquet')['Attrition']

print(f'X_train       {X_train.shape} | attrition {y_train.mean():.3f}')
print(f'X_test        {X_test.shape}  | attrition {y_test.mean():.3f}')
print(f'X_train_smote {X_train_smote.shape} | attrition {y_train_smote.mean():.3f}')

X_train       (1176, 56) | attrition 0.162
X_test        (294, 56)  | attrition 0.160
X_train_smote (1972, 56) | attrition 0.500


## 3. Evaluation helper

`evaluate_model()` is defined locally because helper functions do not carry across stage notebooks. It reports the recall-first metric set — recall, precision and F1 on the positive class, plus ROC-AUC and PR-AUC — at the **default 0.5 threshold**. Moving that threshold to favour recall is a Phase 6 task.

In [4]:
def evaluate_model(model, X, y, name='Model'):
    """Return a dict of recall/precision/F1 (on the positive 'Yes' class),
    ROC-AUC and PR-AUC for a fitted model on (X, y)."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        'model': name,
        'recall_yes': recall_score(y, y_pred),
        'precision_yes': precision_score(y, y_pred),
        'f1_yes': f1_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba),
        'pr_auc': average_precision_score(y, y_proba),
    }

## 4. Imbalance weight for the tree models

The tree models handle the 5.2:1 class imbalance natively rather than by resampling. `scale_pos_weight = #negatives / #positives` is computed from the training labels (≈ 5.19) and fed to XGBoost and LightGBM; Random Forest uses `class_weight='balanced'`, which derives the same idea internally.

In [5]:
# scale_pos_weight = #negatives / #positives, computed from the training labels.
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.3f}  ({(y_train == 0).sum()} No / {(y_train == 1).sum()} Yes)')

scale_pos_weight = 5.189  (986 No / 190 Yes)


## 5. Model specifications

Each model is paired with the imbalance strategy that suits it:

| Model | Imbalance strategy |
|---|---|
| Logistic Regression | SMOTE **inside a pipeline** — resampling is refit on each training fold |
| Random Forest | `class_weight='balanced'` |
| XGBoost | `scale_pos_weight` |
| LightGBM | `scale_pos_weight` |

Wrapping SMOTE in an `imblearn` pipeline is what makes the cross-validation in the next cell leakage-free: the validation fold is always real, untouched data.

The feature matrix is already standardized (the scaler was fit in the feature-engineering stage), so Logistic Regression consumes it directly with no re-scaling.

In [6]:
models = {
    'LogReg (SMOTE)': ImbPipeline([
        ('smote', SMOTE(k_neighbors=5, random_state=RANDOM_STATE)),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'RandomForest': RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        scale_pos_weight=spw, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'LightGBM': LGBMClassifier(
        scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
}

## 6. Cross-validate every model

5-fold `StratifiedKFold` cross-validation on `X_train` — **not** the precomputed SMOTE matrix — scored on recall, F1, ROC-AUC and PR-AUC (`average_precision`, the area under the precision-recall curve). Stratification preserves the ~16% attrition rate in every fold.

The results are sorted by mean recall, then F1, matching the priority order set in Phase 4. This cross-validated comparison — not the test set — is what selects the leader, keeping the test set untouched for Phase 7.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',  # area under the precision-recall curve
}

rows = []
for name, est in models.items():
    cvres = cross_validate(est, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        'model': name,
        'recall': cvres['test_recall'].mean(),
        'recall_std': cvres['test_recall'].std(),
        'f1': cvres['test_f1'].mean(),
        'roc_auc': cvres['test_roc_auc'].mean(),
        'pr_auc': cvres['test_pr_auc'].mean(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values(['recall', 'f1'], ascending=False)
    .reset_index(drop=True)
)
cv_results

,model,recall,recall_std,f1,roc_auc,pr_auc
0,LogReg (SMOTE),0.516,0.091,0.511,0.793,0.562
1,LightGBM,0.437,0.046,0.522,0.808,0.587
2,XGBoost,0.426,0.084,0.520,0.785,0.563
3,RandomForest,0.384,0.120,0.461,0.797,0.551


## 7. Light test-set readout (context only)

For a sanity check, each model is fit on the full training data and read off against the held-out test set. Logistic Regression is fit on the precomputed SMOTE matrix; the tree models on the original training data.

This readout is **indicative only** — it exists to confirm the CV picture transfers to unseen data and to catch anything obviously broken. The authoritative, presentation-grade test evaluation (confusion matrix, curves, final metric table) is Phase 7.

In [8]:
fitted = {}
test_rows = []
for name, est in models.items():
    if name == 'LogReg (SMOTE)':
        # Use the plain LR (no in-pipeline SMOTE) on the precomputed SMOTE matrix.
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(X_train_smote, y_train_smote)
    else:
        model = est
        model.fit(X_train, y_train)
    fitted[name] = model
    test_rows.append(evaluate_model(model, X_test, y_test, name=name))

test_results = (
    pd.DataFrame(test_rows)
    .sort_values(['recall_yes', 'f1_yes'], ascending=False)
    .reset_index(drop=True)
)
test_results

,model,recall_yes,precision_yes,f1_yes,roc_auc,pr_auc
0,LogReg (SMOTE),0.468,0.458,0.463,0.712,0.471
1,LightGBM,0.362,0.586,0.447,0.770,0.500
2,XGBoost,0.340,0.593,0.432,0.748,0.473
3,RandomForest,0.340,0.500,0.405,0.765,0.428


## 8. Select the leader and persist

The leader is the top row of the CV table (recall, then F1). Every fitted model is saved to `processed/models/` so Phase 6 can tune the chosen one without re-running the comparison, alongside:

- `cv_results.parquet` — the cross-validated comparison table
- `model_selection.joblib` — a small record of the leader and the selection metric

Downstream: Phase 6 loads the leader for GridSearch + threshold tuning, Phase 7 evaluates it formally on the test set, and Phase 8 runs SHAP on it (column names were preserved through every stage for exactly this).

In [9]:
leader = cv_results.iloc[0]['model']
print(f'CV leader (recall -> F1): {leader}')

OUT = PROC / 'models'
OUT.mkdir(parents=True, exist_ok=True)

# Persist every fitted model so Phase 6 can tune the leader without re-deciding.
slug = {
    'LogReg (SMOTE)': 'logreg_smote',
    'RandomForest': 'random_forest',
    'XGBoost': 'xgboost',
    'LightGBM': 'lightgbm',
}
for name, model in fitted.items():
    joblib.dump(model, OUT / f'{slug[name]}.joblib')

cv_results.to_parquet(PROC / 'cv_results.parquet', index=False)
joblib.dump({'leader': leader, 'metric': 'cv_recall_then_f1'}, PROC / 'model_selection.joblib')

print(f'Saved {len(fitted)} models to {OUT.resolve()}')
print(f'Saved cv_results.parquet and model_selection.joblib to {PROC.resolve()}')

CV leader (recall -> F1): LogReg (SMOTE)
Saved 4 models to C:\Users\Zeeshan\Documents\Github\Attrition-Prediction-Model\processed\models
Saved cv_results.parquet and model_selection.joblib to C:\Users\Zeeshan\Documents\Github\Attrition-Prediction-Model\processed
